# Parallel

O `dspy.Parallel` permite executar múltiplas chamadas DSPy em paralelo.

Em vez de executar:

Pergunta 1 → aguarda resposta  
Pergunta 2 → aguarda resposta  
Pergunta 3 → aguarda resposta  

podemos executar várias chamadas simultaneamente:

Pergunta 1 ─┐  
Pergunta 2 ─┼─> execução paralela  
Pergunta 3 ─┤  
Pergunta 4 ─┘  

O objetivo principal do `Parallel` é reduzir o tempo total quando existem
várias tarefas independentes.

O `Parallel` não compara nem consolida respostas. Ele apenas executa vários módulos/exemplos
concorrentemente.

In [1]:
import os
from dotenv import load_dotenv
import dspy

load_dotenv()

True

## Setup - Configuração do Modelo

Carregamos as variáveis de ambiente do arquivo `.env` na raiz do projeto. Isso evita hardcoding de credenciais no código.

In [2]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo OpenAI GPT-5 Mini
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Configura o modelo padrão utilizado pelo DSPy
dspy.configure(lm=lm)

## Definindo a tarefa

Vamos criar uma Signature simples para responder diferentes perguntas.

Cada pergunta será independente das demais, o que torna esse cenário
adequado para execução paralela.

In [3]:
class ResponderPergunta(dspy.Signature):
    """Responda a pergunta de maneira objetiva."""

    pergunta: str = dspy.InputField(
        desc="Pergunta que deverá ser respondida"
    )

    resposta: str = dspy.OutputField(
        desc="Resposta objetiva para a pergunta"
    )

In [4]:
resolvedor = dspy.Predict(
    ResponderPergunta
)

## Conjunto de perguntas

Criamos várias tarefas independentes.

Sem `Parallel`, precisaríamos executar cada pergunta individualmente em um
loop, esperando que uma chamada terminasse antes de iniciar a seguinte.

In [5]:
perguntas = [
    "Qual é a capital da Austrália?",
    "Explique em uma frase o que é uma árvore binária.",
    "Quanto é 17 multiplicado por 23?",
    "Qual é a diferença entre uma lista e uma tupla em Python?",
    "O que significa complexidade O(n log n)?",
    "Qual é o maior planeta do Sistema Solar?",
    "Explique em uma frase o que é recursão.",
    "Qual é o valor de 15% de 840?",
]

## Pares `(module, example)`

O `Parallel` espera receber uma lista contendo pares:

(module, example)

Cada par informa:

- qual módulo deve ser executado;
- quais dados devem ser enviados para esse módulo.

Como todas as perguntas utilizarão o mesmo módulo, reutilizamos
`resolvedor` para todos os exemplos.

In [6]:
# Transforma cada pergunta em um dicionário compatível com a Signature
exemplos = [
    {"pergunta": pergunta}
    for pergunta in perguntas
]


# Cria os pares:
#
# (módulo, dados de entrada)
exec_pairs = [
    (resolvedor, exemplo)
    for exemplo in exemplos
]

## Configurando a execução paralela

O parâmetro mais importante é `num_threads`.

Ele determina quantas tarefas podem ser processadas concorrentemente.

Neste exemplo utilizaremos quatro threads.

In [7]:
parallel = dspy.Parallel(
    num_threads=4,              # Até 4 tarefas concorrentes
    max_errors=3,               # Número máximo de erros tolerados
    disable_progress_bar=True  # Exibe barra de progresso
)

resultados = parallel(
    exec_pairs
)

In [8]:
for i, (pergunta, resultado) in enumerate(
    zip(perguntas, resultados),
    start=1
):
    print("=" * 80)
    print(f"PERGUNTA {i}")
    print("=" * 80)

    print(pergunta)

    print("\nResposta:")
    print(resultado.resposta)

    print()

PERGUNTA 1
Qual é a capital da Austrália?

Resposta:
Canberra

PERGUNTA 2
Explique em uma frase o que é uma árvore binária.

Resposta:
Uma árvore binária é uma estrutura de dados hierárquica composta por nós em que cada nó possui no máximo dois filhos, normalmente chamados de filho esquerdo e filho direito.

PERGUNTA 3
Quanto é 17 multiplicado por 23?

Resposta:
391

PERGUNTA 4
Qual é a diferença entre uma lista e uma tupla em Python?

Resposta:
Principais diferenças entre lista e tupla em Python:

- Mutabilidade:
  - Lista: mutável — você pode adicionar, remover ou alterar elementos (ex.: append, pop, insert).
  - Tupla: imutável — não permite alterar sua sequência de elementos após criação.

- Sintaxe:
  - Lista: colchetes — [1, 2, 3]
  - Tupla: parênteses (ou vírgula) — (1, 2, 3) ou (1,) para tupla de um elemento

- Métodos disponíveis:
  - Lista: muitos métodos (append, extend, remove, pop, clear, sort, reverse, etc.).
  - Tupla: poucos métodos (count, index).

- Performance e uso: